# TTAP: five alternatives to Figure 5
The historical figures need no new solver runs. Select option 1–5 below.
The warm-start example is a real **n=1 functional pilot per configuration**, not a paper result.
New experiments are disabled until `RUN_NEW_EXPERIMENT=True`. The objective and generators are unchanged.


In [ ]:
from pathlib import Path
import subprocess, sys
from IPython.display import display, Image
ROOT = next((p for p in [Path.cwd(), Path.cwd() / 'Resuelto_Paper' / 'TTAP_WarmStart'] if (p / 'visual_options.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from Resuelto_Paper/TTAP_WarmStart or the repository root.')
OPTION = '5'  # 1: original + companions; 2: overview; 3: task cards; 4: distributions; 5: original + table
STAT = 'mean'  # or 'median'
FIGURES = ROOT / 'figures' / 'selected'
RUN_NEW_EXPERIMENT = False
NEW_RUN = ROOT / 'runs' / 'new_trace_pilot'
USE_FUNCTIONAL_EXAMPLE = True


In [ ]:
subprocess.run([sys.executable, str(ROOT / 'visual_options.py'), '--option', OPTION, '--stat', STAT, '--output', str(FIGURES)], check=True)
names = {'1': 'option1_original_plus_companions', '2': 'option2_four_large_panels', '3': 'option3_one_card_per_task_count', '4': 'option4_status_and_distributions', '5': 'option5_original_and_numeric_table'}
display(Image(filename=str(FIGURES / (names[OPTION] + '.png'))))
if OPTION == '1':
    for k in (1, 2, 3):
        display(Image(filename=str(FIGURES / f'option1_companion_{k}.png')))


## Separate the greedy plan from solver improvement
A validated greedy route is mapped to the MILP binaries and passed to HiGHS; no decision is fixed.
Compare its final value both to the initial greedy and to the matched cold run with that same greedy retained externally.
A warm-start advantage is not established merely because a plan was already supplied.
The trace graph records actual improving-incumbent callbacks within one run, not a curve reconstructed from independent restarts.


In [ ]:
if RUN_NEW_EXPERIMENT:
    subprocess.run([sys.executable, str(ROOT / 'ttap_warm_start.py'), '--experiments', 'A', 'B', '--tasks', '20', '--seeds', '1', '--budgets', '1', '5', '15', '60', '--trace', '--output', str(NEW_RUN)], check=True)
else:
    print('No new solver experiment launched.')


In [ ]:
if USE_FUNCTIONAL_EXAMPLE:
    run_dir = ROOT / 'examples' / 'functional_pilot_n1_20261004'
    warm_csv = [run_dir / 'metrics.csv']
    extra = ['--allow-partial', '--trace-csv', str(run_dir / 'trace.csv'), '--trace-scenario', 'A4x4_T20_S1', '--trace-budget', '60']
elif RUN_NEW_EXPERIMENT:
    warm_csv = [NEW_RUN / 'metrics.csv']
    extra = ['--allow-partial', '--trace-csv', str(NEW_RUN / 'trace.csv'), '--trace-scenario', 'A4x4_T20_S1', '--trace-budget', '60']
else:
    warm_csv = [ROOT / 'runs' / 'A_full' / 'metrics.csv', ROOT / 'runs' / 'B_full' / 'metrics.csv']
    extra = []  # Full-cohort validation remains enabled.
WARM_FIGURES = ROOT / 'figures' / 'warm_selected'
subprocess.run([sys.executable, str(ROOT / 'warm_visual_options.py'), '--warm-csv', *map(str, warm_csv), *extra, '--output', str(WARM_FIGURES)], check=True)
for name in ['warm1_greedy_and_higher_budget_plans', 'warm2_credit_to_greedy_vs_injection', 'warm3_setup_cost_and_quality', 'warm4_true_incumbent_trace']:
    path = WARM_FIGURES / (name + '.png')
    if path.exists():
        display(Image(filename=str(path)))
